# WildSpeech M8.5 · M0+ Strong Screening Baseline

**Milestone:** Improve the reference using only *observable screening, item scores and administrative metadata*. Do not use audio in Phase 1. Reference: M8.4 mean fold R² **0.193227**, mean pooled OOF R² **0.199113**, school disjoint R² **0.147363**.

**Execute SCREEN first** for infrastructure and figures, then edit MODE to FULL and run this notebook again to execute all 100 frozen folds. SCREEN is **not promotion eligible**. FULL preregisters E00–E09 and uses M8.4's existing fold hash and OOF as an immutable anchor.

No real-data results are bundled with this notebook. Do not submit to the leaderboard unless FULL promotion gates pass. Never commit row-level child data or OOF predictions.

## 0. Mount Google Drive and identify dataset (no GPU required)

The official six CSVs and original M8.4 `phase0` run must already exist in Drive. This notebook does not download, re-extract, or retrain audio.

In [ ]:
import os,sys,json,subprocess,importlib,time
from pathlib import Path
from IPython.display import display,Markdown,Image
import pandas as pd
import numpy as np
try:
    IN_COLAB=importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB=False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive",force_remount=False)
    DRIVE=Path("/content/drive/MyDrive")
else:
    DRIVE=Path(os.environ.get("WILDSPEECH_DRIVE_ROOT",str(Path.home()/"MyDrive")))
assert DRIVE.is_dir(),f"Drive unavailable: {DRIVE}"
print("✓ Drive:",DRIVE)

## 1. Install the GitHub source

This uses the editable local package. Git source SHA is logged with every experimental run. Never use a stale manually copied module.

In [ ]:
URL="https://github.com/mybenkhadda/WildSpeech.git"
CHECKOUT=Path("/content/WildSpeech_git") if IN_COLAB else Path.cwd()/"WildSpeech_git"
if not (CHECKOUT/".git").exists():
    if CHECKOUT.exists():raise RuntimeError(f"Unrecognized existing directory: {CHECKOUT}")
    subprocess.run(["git","clone","--depth","1","--branch","main",URL,str(CHECKOUT)],check=True)
else:
    subprocess.run(["git","-C",str(CHECKOUT),"pull","--ff-only","origin","main"],check=True)
PKG=CHECKOUT/"WildSpeech_M8"
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(PKG)],check=True)
SOURCE=str(PKG/"src")
if SOURCE not in sys.path:sys.path.insert(0,SOURCE)
from wildspeech_m85.experiments import execute,locked_plan,EVAL_HASH_2026,CANDIDATES
from wildspeech_m84.data import locate_data_root,load_official
from wildspeech_m84.metrics import summarize_oof
GIT_SHA=subprocess.check_output(["git","-C",str(CHECKOUT),"rev-parse","HEAD"],text=True).strip()
print("✓ M8.5 installed from:",GIT_SHA)

## 2. Mode and output configuration

`SCREEN`: only repeat 0 (5 student folds + 5 held-out-school folds), one CatBoost seed and 250 trees, no promotion. `FULL`: locked 50+50 folds, five seeds and 600 iterations, all model/ablation arms and formal decision gates.

**Keep frozen Phase 0 outputs untouched.**

In [ ]:
MODE="SCREEN"                         # change to "FULL" after checking SCREEN
RUN_MODELS=list(CANDIDATES)            # E01–E09, preregistered
DATA_ROOT_OVERRIDE=""
PHASE0_ROOT=DRIVE/"WildSpeech_M84"/"phase0"
OUTPUT_ROOT=DRIVE/"WildSpeech_M85"/"m0plus"
EXPECTED_FOLD_HASH="6ee0f59c8cf5ea3d8f20f5fd500353419587660e00949ac6f1e6a16e092c9ce1"
assert MODE in ("SCREEN","FULL")
assert (PHASE0_ROOT/"folds"/"folds.parquet").is_file(),"M8.4 frozen folds missing"
assert (PHASE0_ROOT/"parity"/"M84_RESULTS.json").is_file(),"M8.4 reference report missing"
print("Mode:",MODE,"| Private output:",OUTPUT_ROOT)
print("Phase 0 root (READ ONLY):",PHASE0_ROOT)

## 3. Original Phase 0 integrity and screening metadata audit

Before new fitting, verify training identity, missing tasks, exact reference parity report, test item-score availability and that no row-level test statistics are fitted.

In [ ]:
DATA_ROOT=locate_data_root(DRIVE,DATA_ROOT_OVERRIDE)
DATA=load_official(DATA_ROOT)
P0=json.loads((PHASE0_ROOT/"parity"/"M84_RESULTS.json").read_text())
assert P0["parity_pass"],"Reference parity did not pass"
assert P0["fold_hash"]==EXPECTED_FOLD_HASH,"Reference folds changed"
print("Reference frozen mean-fold R²:",P0["reference_metrics"]["mean_fold_r2"])
print("Reference frozen student pooled R²:",P0["reference_metrics"]["mean_pooled_oof_r2"])
print("Students:",len(DATA.train_students),"/",len(DATA.test_students))
print("Clips:",len(DATA.train_clips),"/",len(DATA.test_clips))
print("Test item_score present:","item_score" in DATA.test_clips)
print("Test item_score observed share:",float(DATA.test_clips.item_score.notna().mean()) if "item_score" in DATA.test_clips else None)
print("Phase 1 features do NOT use score_status or school_id as predictors.")

## 4. Preregistration preview — E00 through E09

Each arm has a fixed model family and feature block contract. The experiment engine writes the immutable preregistration and run ledger **before** any new model fit.

In [ ]:
PREDECLARED=pd.DataFrame(locked_plan())
display(PREDECLARED[["comparison_id","baseline","challenger","scope"] if "scope" in PREDECLARED else ["comparison_id","baseline","challenger"]])
assert "E04_VS_E00" in PREDECLARED.comparison_id.values
print("Predeclared comparisons:",len(PREDECLARED))

## 5. Execute complete M8.5 experiment engine

Reuses immutable M8.4 reference OOF and original folds; performs training-only transformations and nested Ridge alpha tuning, nested S0 blend, CatBoost, bundled S2–S5 and leave-one-block-out ablations.

Every fold is checkpointed. Interrupted full executions can resume after an unchanged checkout. Full model selection never reads the outer-validation outcomes when fitting features or blend weights.

In [ ]:
started=time.perf_counter()
RESULT=execute(
    drive_root=DRIVE,
    data_root_override=DATA_ROOT_OVERRIDE,
    phase0_root_override=str(PHASE0_ROOT),
    output_root_override=str(OUTPUT_ROOT),
    mode=MODE,
    run_models=tuple(RUN_MODELS),
    expected_fold_hash=EXPECTED_FOLD_HASH,
    export_final=True,
)
print("✓ Completed/reused",MODE,"in",round(time.perf_counter()-started,1),"s")
print("Decision:",RESULT["decision"],"| selected:",RESULT.get("winner"))
print("Reference OOF SHA:",RESULT["reference_oof_sha256"])
print("Private report:",OUTPUT_ROOT/"reports"/"M85_M0PLUS_DECISION.md")

## 6. Screening hypothesis atlas: school × instrument transfer

School ID is used **only** for audits and held-out-school splitting. Suppress cells below five in shared figures; do not publish private school tables.

In [ ]:
path=OUTPUT_ROOT/"diagnostics"
for filename in ("instrument_transfer_by_repeat.csv","missingness_by_instrument.csv",
                 "task_availability_train_test_coverage_only.csv"):
    print("\n",filename)
    display(pd.read_csv(path/filename).head(14))
print("\nItem feature contract:")
ITEM_CONTRACT=json.loads((path/"item_contract.json").read_text())
display(pd.DataFrame([{"key":k,"value":str(v)[:150]} for k,v in ITEM_CONTRACT.items()]))
for filename in ("01_school_instrument_counts_private.png","02_school_instrument_transfer.png",
                 "03_missingness_by_instrument.png"):
    figure=OUTPUT_ROOT/"figures"/filename
    if figure.exists():display(Image(filename=str(figure)))

## 7. Frozen-fold R² and paired model comparison

Primary selection uses mean of 10-repeat pooled student R², while the gate uses mean paired fold ΔR² on all 50 student folds. Never compare individual folds with different split hashes.

In [ ]:
RESULTS=pd.DataFrame(RESULT["model_metrics"])
cols=[c for c in ["model","mean_fold_r2","mean_student_pooled_r2",
                  "mean_school_pooled_r2","student_mae","student_rmse",
                  "mean_fold_delta_r2","corrected_t","win_fraction",
                  "school_pooled_delta_r2","gate"] if c in RESULTS]
display(RESULTS[cols].round(5))
if MODE=="FULL":
    assert all(RESULTS[RESULTS.model!="E00"].n_folds==100)
else:
    print("SCREEN exploratory-only results: no corrected significance or promotion.")
for filename in ("04_candidate_pooled_r2.png","05_paired_delta_r2.png"):
    f=OUTPUT_ROOT/"figures"/filename
    if f.exists():display(Image(filename=str(f)))

## 8. Feature-family ablations and nuisance controls

E04 bundles S2/S3/S4/S5 together. E05–E08 remove one block conditionally. E09 removes screening protocol nuisance proxies (reference duration and sample-rate fields). This does not claim audio is useful.

In [ ]:
if MODE=="FULL":
    PAIRS=pd.read_csv(OUTPUT_ROOT/"tables"/"paired_comparison_audit.csv")
    display(PAIRS.round(5))
    print("Explanation only: E05–E09 cannot independently become the chosen model.")
else:
    print("Only FULL has 50-fold corrected paired comparisons.")

## 9. Formal M0+ decision and optional candidate submission

A new candidate submission is exported **only** when FULL completed all preregistered comparisons and an eligible E01–E04 model passed every gate. Otherwise the exact M8.4 reference remains the M0+ fallback.

In [ ]:
display(Markdown((OUTPUT_ROOT/"reports"/"M85_M0PLUS_DECISION.md").read_text()))
if RESULT["decision"]=="PROMOTE":
    f=OUTPUT_ROOT/"submission"/"m0plus_submission.csv"
    print("✓ Candidate prediction file (NOT uploaded):",f)
    manifest=json.loads((OUTPUT_ROOT/"submission"/"submission_manifest.json").read_text())
    print("SHA-256:",manifest["sha256"])
    assert manifest["singleton_raw_pipeline_test"]
    assert manifest["test_permutation_invariance"]
else:
    print("No new challenger submission exported; reference remains available from Phase 0.")

## 10. Ledger and full execution check

Validate run hashes and private outputs. Changing feature code produces a new run identity; it never silently overwrites the old reference or candidate artifacts.

In [ ]:
ledger=pd.read_parquet(OUTPUT_ROOT/"ledger"/"experiments.parquet")
display(ledger[[c for c in ["model_id","status","git_sha","fold_hash","mean_fold_r2","mean_pooled_oof_r2"] if c in ledger]].tail(15))
if MODE=="FULL":
    assert RESULT["completed_all_E01_E09"] and RESULT["decision"] in ("PROMOTE","HOLD_REFERENCE")
print("Frozen M8.4 split hash preserved:",RESULT["fold_hash"]==EXPECTED_FOLD_HASH)
print("Report and private results:",OUTPUT_ROOT/"reports")

## 11. Phase 1 handoff

- **PROMOTE:** freeze the submitted M0+ model and its SHA; use it as the compulsory base for future genuine-speech experiments.
- **HOLD_REFERENCE:** preserve the original 14-feature Ridge reference; the M0+ experiment remains scientifically useful even without a gain.
- **SCREEN_EXPLORATORY:** complete the 100-fold FULL protocol before making conclusions.

The next milestone is Phase 2 speech extraction on separate GPU caches, with per-clip frozen-model outputs and no training on test rows.